# 🚀 Freelancer System Testing - Interactive Notebook

Este notebook demonstra todos os 7 agentes + 3 componentes de aprendizado do módulo freelancer.

**Requisitos:**
- Backend rodando (não precisa de frontend)
- Database com dados de teste
- Bibliotecas instaladas: `pip install jupyter requests pandas`

In [ ]:
# Setup
import sys
from pathlib import Path

# Add backend to path
backend_path = Path.cwd().parent
sys.path.insert(0, str(backend_path))

from database.connection import SessionLocal
from database.models import User, PricingParameter, FreelancePlatform
from services.freelancer import (
    ProjectDuplicationPrevention,
    FreelancerFinancialCalculator,
    ClientRiskAssessment,
    PricingLearner,
    RejectionPatternLearner,
    HourlyRateOptimizer,
    create_integration_service,
)

# Create database session
db = SessionLocal()
print("✓ Database connection established")

## 1. 🧩 RN09: Duplication Prevention

Previne projetos duplicados usando similaridade de Jaccard.

In [ ]:
dup_service = ProjectDuplicationPrevention(db)

# Test similarity
is_duplicate = dup_service.is_duplicate(
    title="Python Backend API Development",
    description="Need a Python FastAPI developer for building RESTful APIs",
    platform_id=1,
    external_id="test_001",
)

print(f"Is duplicate: {is_duplicate}")
print(f"Similarity threshold: {dup_service.similarity_threshold}")

## 2. 💰 RN11: Financial Calculator

Calcula USD → BRL + impostos brasileiros (Simples Nacional, MEI, etc.)

In [ ]:
calc = FreelancerFinancialCalculator(db, user_id=1)

result = calc.calculate_net_income(
    gross_usd=5000.0,
    platform="upwork",
    tax_regime="simples_nacional",
    include_breakdown=True,
)

import pandas as pd

# Display as DataFrame
df = pd.DataFrame([
    {"Item": "Gross (USD)", "Value": f"${result['gross_usd']:,.2f}"},
    {"Item": "Exchange Rate", "Value": f"R$ {result['exchange_rate']:.2f}"},
    {"Item": "Gross (BRL)", "Value": f"R$ {result['gross_brl']:,.2f}"},
    {"Item": "Platform Fee (10%)", "Value": f"R$ {result['breakdown']['platform_fee_brl']:,.2f}"},
    {"Item": "Tax (Simples)", "Value": f"R$ {result['breakdown']['simples_nacional_tax_brl']:,.2f}"},
    {"Item": "Net Income (BRL)", "Value": f"R$ {result['net_brl']:,.2f}"},
    {"Item": "Effective Tax Rate", "Value": f"{result['effective_tax_rate'] * 100:.2f}%"},
])

df

## 3. ⚖️ RN12: Client Risk Assessment

Avalia risco do cliente (red flags, green flags, risk score).

In [ ]:
risk_service = ClientRiskAssessment(db, user_id=1)

# Test high-risk client
high_risk = risk_service.assess_risk(
    client_rating=2.5,
    client_projects_count=2,
    project_description="Need this ASAP! Budget is flexible but looking for cheapest option.",
    client_payment_verified=False,
    client_country="Unknown",
)

# Test low-risk client
low_risk = risk_service.assess_risk(
    client_rating=4.8,
    client_projects_count=50,
    project_description="Looking for experienced Python developer for 3-month project.",
    client_payment_verified=True,
    client_country="United States",
)

# Compare
comparison = pd.DataFrame([
    {
        "Client Type": "High Risk",
        "Risk Score": f"{high_risk['risk_score']:.1f}/10",
        "Risk Level": high_risk['risk_level'],
        "Red Flags": len(high_risk['red_flags']),
        "Green Flags": len(high_risk['green_flags']),
        "Recommendation": high_risk['recommendation'],
    },
    {
        "Client Type": "Low Risk",
        "Risk Score": f"{low_risk['risk_score']:.1f}/10",
        "Risk Level": low_risk['risk_level'],
        "Red Flags": len(low_risk['red_flags']),
        "Green Flags": len(low_risk['green_flags']),
        "Recommendation": low_risk['recommendation'],
    },
])

comparison

## 4. 🧠 Learning Component: PricingLearner

Aprende com projetos executados e ajusta parâmetros de precificação.

In [ ]:
pricing_learner = PricingLearner(db, user_id=1)

# Analyze performance
performance = pricing_learner.analyze_pricing_performance(days=90)

if performance['total_records'] > 0:
    print(f"Total learning records: {performance['total_records']}")
    print(f"Average accuracy: {performance['avg_accuracy_score']:.2%}")
    print(f"Average error margin: {performance['avg_error_margin']:.2%}")
    print(f"Needs adjustment: {performance['needs_adjustment']}")
    
    # Show complexity performance
    if performance['complexity_performance']:
        complexity_df = pd.DataFrame([
            {"Range": k, **v} 
            for k, v in performance['complexity_performance'].items()
        ])
        display(complexity_df)
else:
    print(f"No learning records found: {performance['message']}")

## 5. 🔍 Learning Component: RejectionPatternLearner

Analisa quais red flags realmente predizem rejeição.

In [ ]:
rejection_learner = RejectionPatternLearner(db, user_id=1)

# Analyze patterns
patterns = rejection_learner.analyze_rejection_patterns(days=90)

if patterns['total_opportunities'] > 0:
    print(f"Total opportunities: {patterns['total_opportunities']}")
    print(f"Rejected: {patterns['rejected_count']}")
    print(f"Accepted: {patterns['accepted_count']}")
    print(f"Rejection rate: {patterns['rejection_rate']:.1%}")
    
    # Show high-risk flags
    if patterns['high_risk_flags']:
        print("\nHigh-risk red flags (>70% rejection):")
        high_risk_df = pd.DataFrame(patterns['high_risk_flags'])
        display(high_risk_df)
    
    # Show false positives
    if patterns['false_positive_flags']:
        print("\nFalse positive flags (<40% rejection):")
        false_positive_df = pd.DataFrame(patterns['false_positive_flags'])
        display(false_positive_df)
else:
    print(f"No opportunities found: {patterns['message']}")

## 6. 📊 Learning Component: HourlyRateOptimizer

Otimiza taxa horária baseado em acceptance patterns.

In [ ]:
rate_optimizer = HourlyRateOptimizer(db, user_id=1)

# Analyze acceptance by rate
analysis = rate_optimizer.analyze_acceptance_by_rate(days=90)

if analysis['total_opportunities'] > 0:
    print(f"Total opportunities analyzed: {analysis['total_opportunities']}")
    
    # Show rate range stats
    rate_stats_df = pd.DataFrame([
        {"Range": k, **v}
        for k, v in analysis['rate_range_stats'].items()
        if v['total'] > 0
    ])
    display(rate_stats_df)
    
    # Show optimal range
    if analysis['optimal_range']:
        optimal = analysis['optimal_range']
        print(f"\n✨ Optimal range: {optimal['range_name']}")
        print(f"   Rate: ${optimal['range_min']}-${optimal['range_max']}/hr")
        print(f"   Expected value: ${optimal['expected_value']:.2f}")
        print(f"   Acceptance rate: {optimal['acceptance_rate']:.1%}")
else:
    print(f"No opportunities found: {analysis['message']}")

## 7. 🔗 Integration Service Test

Testa pipeline completo (RN09-RN13 + learning components).

In [ ]:
integration_service = create_integration_service(db, user_id=1)

# Process opportunity through full pipeline
result = integration_service.process_opportunity(
    title="AI/ML Model Development",
    description="Looking for ML engineer to build recommendation system using PyTorch",
    client_budget=8000.0,
    platform_name="upwork",
    external_id="upwork_ml_001",
    client_rating=4.5,
    client_projects_count=25,
    client_payment_verified=True,
)

print("Full Pipeline Results:")
print(f"  - Is duplicate: {result['is_duplicate']}")
print(f"  - Risk level: {result['risk_assessment']['risk_level']}")
print(f"  - Risk score: {result['risk_assessment']['risk_score']:.1f}/10")
print(f"  - Net income (BRL): R$ {result['financial_calculation']['net_brl']:,.2f}")
print(f"  - Rate limit: {result['rate_limit_status']['requests_remaining']} remaining")

# Display full result
import json
print("\nFull result:")
print(json.dumps(result, indent=2, default=str))

## ✅ Cleanup

Feche a conexão do banco quando terminar.

In [ ]:
db.close()
print("✓ Database connection closed")